# Homework 2. Machine Learning in Economics

Welcome to Homework 2 of Machine Learning in Economics! In this homework, you'll get hands-on practice coding and interpreting neural networks with two widely used Python packages, PyTorch and TensorFlow/Keras.

The structure is the same as in Homework 1. Run the supplied baselines, interpret their outputs, and make one small change in each extension. Write your responses in the answer cells and place any additional code immediately below the relevant question.

The focus is on learning. A thoughtful attempt that shows what you learned is enough for the homework to count toward the bonus, even if your model's performance is imperfect. If you complete two of the three homeworks, you'll receive a **+0.5 bonus to your project grade**.

Please email your completed Jupyter notebook (`.ipynb`) to paul.stephan@econ.uzh.ch. Submit one notebook per group of one to four students. The hand-in date is 17th of November (23:59).

Enjoy learning about neural networks!

### Getting started

- Open this notebook in Google Colab or Jupyter. Run cells in order with **Shift + Enter**. The first run downloads two datasets, so you need internet access. A CPU runtime is sufficient.
- **PyTorch** builds the credit model, **Fairlearn** calculates group metrics, and **TensorFlow/Keras** builds the image model. NumPy and pandas handle arrays and tables, scikit-learn handles splitting and evaluation, Matplotlib draws plots, and xlrd lets pandas read the credit workbook.
- If an import fails, uncomment the relevant installation line below and run it. Restart the kernel if prompted, then run the notebook from the top. If a variable is undefined, rerun the cells that create it.
- Use validation data to choose settings and thresholds. Use test data after making your choices. Restart the kernel and rerun all cells before submitting the completed notebook.

In [ ]:
# Uncomment only the line for the packages you need.
# %pip install numpy pandas matplotlib scikit-learn
# %pip install torch fairlearn xlrd
# %pip install tensorflow

## Part 1. Consumer Credit Distress and Access to Debt Counseling

### Economic context

Can repayment histories help identify consumers at risk of missing a payment? Imagine a public debt-counseling program considering voluntary, confidential outreach. Missing someone who subsequently defaults could mean a missed opportunity to offer support. Contacting someone who would repay anyway uses staff time and may impose privacy or stigma costs.

You will use records from Taiwan in 2005 to study next-month default prediction and the distribution of outreach across groups. The data describe existing cardholders and their repayment under historical conditions. Estimating whether outreach prevents default would require evidence on the intervention, such as a randomized pilot.

The [UCI Default of Credit Card Clients dataset](https://doi.org/10.24432/C55S3H) contains 30,000 records and 23 candidate predictors, excluding the identifier and outcome.

### Variable definitions

- `ID`: Client identifier.

- `LIMIT_BAL`: Amount of the given credit (NT dollars). Includes both individual consumer credit and family (supplementary) credit.

- `SEX`: Recorded sex (`1` = male; `2` = female).

- `EDUCATION`: Education (`1` = graduate school; `2` = university; `3` = high school; `4` = other). Inspect additional codes in the raw data.

- `MARRIAGE`: Marital status (`1` = married; `2` = single; `3` = others).

- `AGE`: Age (years).

### Repayment status history

These variables record repayment status by month, with the following coding:
- `-2` = an additional code in the raw data
- `-1` = pay duly/pay on time
- `0` = an additional code in the raw data
- `1` = payment delay for one month
- `2` = payment delay for two months
- …
- `8` = payment delay for eight months
- `9` = payment delay for nine months and above

Month mapping (most recent month is September 2005):
- `PAY_0`: Repayment status this month
- `PAY_2`: Repayment status last month
- `PAY_3`: Repayment status two months ago
- `PAY_4`: Repayment status three months ago
- `PAY_5`: Repayment status four months ago
- `PAY_6`: Repayment status five months ago

### Bill statement amounts

Month mapping (most recent month is September 2005):
- `BILL_AMT1`: Bill statement amount this month
- `BILL_AMT2`: Bill statement amount last month
- `BILL_AMT3`: Bill statement amount two months ago
- `BILL_AMT4`: Bill statement amount three months ago
- `BILL_AMT5`: Bill statement amount four months ago
- `BILL_AMT6`: Bill statement amount five months ago

### Previous payment amounts

Month mapping (most recent month is September 2005):
- `PAY_AMT1`: Amount paid this month
- `PAY_AMT2`: Amount paid last month
- `PAY_AMT3`: Amount paid two months ago
- `PAY_AMT4`: Amount paid three months ago
- `PAY_AMT5`: Amount paid four months ago
- `PAY_AMT6`: Amount paid five months ago

### Outcome

- `default payment next month`: Indicator for whether the client defaults on payment in the next month (binary outcome).

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from fairlearn.metrics import (
    MetricFrame,
    selection_rate, true_positive_rate, false_positive_rate,
    demographic_parity_difference, demographic_parity_ratio,
    equalized_odds_difference, equalized_odds_ratio
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

data_file = Path("default of credit card clients.xls")
if not data_file.exists():
    url = "https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip"
    with urlopen(url, timeout=60) as response, ZipFile(BytesIO(response.read())) as archive:
        data_file.write_bytes(archive.read(data_file.name))

# Read the excel file, skipping the first row (metadata) and using the second row (index 1) as header.
credit_df = pd.read_excel(data_file, header=1)

# Extract features (X) and target (y).
# Drop the 'ID' column and the 'default payment next month' column from the features.
X = credit_df.drop(columns=['ID', 'default payment next month']).copy()
# The target variable is 'default payment next month'.
y = credit_df['default payment next month'].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Columns:", list(X.columns)[:10], "...")
print("y value counts:", pd.Series(y).value_counts().head())

In [ ]:
X.head()

### Recorded sex and the outreach audit

The supplied baseline includes `SEX` as a predictor so you can investigate the consequences of using it. We also retain it separately to measure group disparities. In the audit, consider whether other features could carry related information.

In [ ]:
A_sex = X["SEX"]

In [ ]:
# Train/test/validation splitting
X_trainval, X_test, y_trainval, y_test, A_trainval, A_test = train_test_split(
    X, y, A_sex, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

X_train, X_val, y_train, y_val, A_train, A_val = train_test_split(
    X_trainval, y_trainval, A_trainval, test_size=0.25, random_state=RANDOM_SEED, stratify=y_trainval
)  # => 60% train, 20% val, 20% test

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Default rate (train/val/test):", y_train.mean(), y_val.mean(), y_test.mean())

### Preparing data for PyTorch

A **tensor** is an array that PyTorch can use for model fitting. Here, the feature tensor has one row per cardholder and one column per feature. The label tensor has one column because the model produces one score per cardholder. A `Dataset` supplies a feature row and its label, and a `DataLoader` groups these pairs into batches. The training loader shuffles rows each epoch. The validation and test loaders preserve row order so scores can be matched to group labels.

**Coding tip.** `next(iter(train_loader))` returns the first batch as `(features, labels)`. Check each tensor's `.shape` to see what the model receives. [PyTorch Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)

In [ ]:
# Standardizing data is important for NN training
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# Some preliminary setup before defining the NN architecture
class TabDataset(Dataset):
    def __init__(self, X_np, y_np):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.float32).view(-1, 1)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = TabDataset(X_train_s, y_train.to_numpy())
val_ds   = TabDataset(X_val_s, y_val.to_numpy())
test_ds  = TabDataset(X_test_s, y_test.to_numpy())

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=512, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=512, shuffle=False)

**Question 1.** Explain how the code prepares the data for neural network training. Why is the scaler fitted on the training observations alone? Using the variable definitions above, identify features for which standardizing the recorded codes may be a poor representation. Propose and justify an alternative encoding.

*Your answer:*

### Building the feedforward model

`nn.Sequential` applies layers in the order listed. `nn.Linear` takes an input width and an output width. `ReLU` adds nonlinearity, and `Dropout` randomly zeroes some activations during training. The last layer returns a raw score called a **logit**. The loss accepts logits directly, and `torch.sigmoid` converts them to probabilities when making predictions. Try `print(model)` to inspect the layers.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)  # logits
        )
    def forward(self, x):
        return self.net(x)

model = MLP(d_in=X_train_s.shape[1])

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

---

**Question 2.** Briefly explain the architecture of the baseline model. Feel free to use a schematic.


---

*Your answer:*

### Training and early stopping

An **epoch** is one pass through the fitting data. For each batch, the loop clears old gradients with `zero_grad()`, computes predictions and loss, calculates gradients with `backward()`, and updates weights with `step()`. After each epoch it checks validation AUC and saves the best weights. The loop stops after three epochs without a sufficient improvement and reloads those saved weights.

**Coding tip.** `model.train()` activates dropout for fitting. `model.eval()` turns it off for scoring, while `torch.no_grad()` skips gradient tracking.

In [ ]:
def predict_proba_torch(model, loader):
    model.eval()
    ps = []
    ys = []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            p = torch.sigmoid(logits)
            ps.append(p.numpy().ravel())
            ys.append(yb.numpy().ravel())
    return np.concatenate(ys), np.concatenate(ps)

best_state = None
best_val_auc = -np.inf
patience = 3
pat = 0

for epoch in range(1, 21):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    yv, pv = predict_proba_torch(model, val_loader)
    val_auc = roc_auc_score(yv, pv)
    print(f"Epoch {epoch:02d} | val AUC = {val_auc:.4f}")

    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= patience:
            print("Early stopping.")
            break

model.load_state_dict(best_state)

In [ ]:
yv, pv = predict_proba_torch(model, val_loader)
print("DNN ROC AUC (validation):", roc_auc_score(yv, pv))

### A loss function for outreach

Assign one loss unit to an unnecessary outreach contact and ten to a missed contact with someone who subsequently defaults. Correct classifications have zero loss in this stylized exercise. The weights express an assumed social trade-off.

**Question 3.** Define the false positive and false negative in this setting. Who bears the consequences of each? Discuss how evidence on program costs, take-up, and treatment effects could inform the assumed weights. Explain what changes in the decision problem if some high-risk consumers benefit little from an offer.

*Your answer:*

In [ ]:
# Helper functions to evaluate expected costs

def expected_cost(y_true, y_hat, cost_fn=10.0, cost_fp=1.0):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    return cost_fn*fn + cost_fp*fp

def choose_threshold_by_cost(y_true, p, cost_fn=10.0, cost_fp=1.0, grid=None):
    if grid is None:
        grid = np.linspace(0.01, 0.99, 99)
    best = None
    for t in grid:
        y_hat = (p >= t).astype(int)
        c = expected_cost(y_true, y_hat, cost_fn=cost_fn, cost_fp=cost_fp)
        if (best is None) or (c < best["cost"]):
            best = {"t": t, "cost": c}
    return best

In [ ]:
best_dnn = choose_threshold_by_cost(yv, pv, cost_fn=10.0, cost_fp=1.0)
t_dnn = best_dnn["t"]
yhat_val_dnn = (pv >= t_dnn).astype(int)

print("Chosen threshold (DNN):", t_dnn)
print("Validation confusion matrix (DNN):\n", confusion_matrix(yv, yhat_val_dnn))
print("Total validation loss (DNN):", expected_cost(yv, yhat_val_dnn))

**Question 4.** Interpret the threshold, confusion matrix, and total loss in terms of outreach. Compare loss per person with the rules that contact everyone or contact no one. Why might a model with a higher ROC AUC have higher loss at a particular threshold?

*Your answer:*

### Permutation feature importance

First, measure the fitted model's AUC on the validation data. Then shuffle one feature across people while leaving all other features and outcomes in place. Score the same model again without retraining. The code averages the drop in AUC over repeated shuffles. A larger drop means the model relied more on that feature for its predictions in this sample. Correlated features can share predictive information, and a feature's importance does not measure its causal effect on default.

In [ ]:
feature_names = list(X.columns)

def dnn_predict_proba_from_numpy(Xs_np):
    # Xs_np must already be scaled
    loader = DataLoader(TabDataset(Xs_np, np.zeros(len(Xs_np))), batch_size=1024, shuffle=False)
    model.eval()
    ps = []
    with torch.no_grad():
        for xb, _ in loader:
            ps.append(torch.sigmoid(model(xb)).numpy().ravel())
    return np.concatenate(ps)

def permutation_importance_auc(model_predict_proba_fn, Xs, y_true, n_repeats=3, seed=0):
    rng = np.random.default_rng(seed)
    base = roc_auc_score(y_true, model_predict_proba_fn(Xs))
    importances = []
    for j in range(Xs.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = Xs.copy()
            rng.shuffle(Xp[:, j])
            auc_p = roc_auc_score(y_true, model_predict_proba_fn(Xp))
            drops.append(base - auc_p)
        importances.append(np.mean(drops))
    return base, np.array(importances)

base_auc, imps = permutation_importance_auc(dnn_predict_proba_from_numpy, X_val_s, y_val.to_numpy(), n_repeats=2, seed=RANDOM_SEED)

imp_df = pd.DataFrame({"feature": feature_names, "auc_drop": imps}).sort_values("auc_drop", ascending=False)
imp_df.head(15)

**Question 5.** Identify the five largest drops in validation AUC. For each, suggest what it could capture about household finances or past access to credit. Explain how correlated repayment histories could affect the ranking. Identify a possible proxy for a sensitive attribute and explain what evidence would support that interpretation. What would you need to estimate a causal effect on default?

*Your answer:*

### Fairness assessment with Fairlearn

`MetricFrame` takes observed labels, predicted decisions, and a group label for each row. `.by_group` returns one row per group. Here the selection rate is the share flagged for outreach, TPR is the share flagged among people who default, and FPR is the share flagged among people who repay.

**Coding tip.** Check that the three inputs have the same number of rows and preserve the same ordering. Resetting indices gives the inputs common row labels. [Fairlearn assessment guide](https://fairlearn.org/main/user_guide/assessment/perform_fairness_assessment.html)

In [ ]:
# Use the DNN threshold chosen earlier
A_val_series = pd.Series(A_val).reset_index(drop=True)
y_val_series = pd.Series(y_val).reset_index(drop=True)

# Align arrays
p_val_dnn = pd.Series(pv).reset_index(drop=True)
yhat_val_dnn = (p_val_dnn >= t_dnn).astype(int)

metrics = {
    "selection_rate": selection_rate,
    "TPR": true_positive_rate,
    "FPR": false_positive_rate,
}

mf = MetricFrame(
    metrics=metrics,
    y_true=y_val_series,
    y_pred=yhat_val_dnn,
    sensitive_features=A_val_series
)

print(mf.by_group)

# ---- Summary fairness metrics (demographic parity + equalized odds) ----
dp_diff  = demographic_parity_difference(y_val_series, yhat_val_dnn, sensitive_features=A_val_series)
dp_ratio = demographic_parity_ratio(y_val_series, yhat_val_dnn, sensitive_features=A_val_series)

eo_diff  = equalized_odds_difference(y_val_series, yhat_val_dnn, sensitive_features=A_val_series)
eo_ratio = equalized_odds_ratio(y_val_series, yhat_val_dnn, sensitive_features=A_val_series)

print("\nFairness summary metrics (across groups):")
print(f"Demographic parity difference: {dp_diff:.4f}")
print(f"Demographic parity ratio:  {dp_ratio:.4f}")
print(f"Equalized odds difference:  {eo_diff:.4f}")
print(f"Equalized odds ratio:  {eo_ratio:.4f}")

**Question 6.** Interpret the validation audit. What do selection rate, true positive rate, and false positive rate mean for access to counseling and the burden of unnecessary contact? Define demographic parity and equalized odds. Compare the metrics and explain how differences in observed default rates could affect them. Which disparities would you prioritize for this program, and why?

*Your answer:*

### Matching true positive rates

This example chooses thresholds by recorded sex to bring each group's true positive rate close to the baseline's overall rate. Both fitting and illustration use validation data here. Assess any rule you carry forward on the held-out test set after freezing its thresholds.

In [ ]:
def threshold_to_hit_tpr(y_true, p, target_tpr):
    # search over thresholds; pick one with closest TPR
    grid = np.linspace(0.01, 0.99, 99)
    best = None
    for t in grid:
        yhat = (p >= t).astype(int)
        tpr = true_positive_rate(y_true, yhat)
        if best is None or abs(tpr - target_tpr) < best["gap"]:
            best = {"t": t, "tpr": tpr, "gap": abs(tpr - target_tpr)}
    return best

# Choose a target TPR = overall TPR of the original DNN threshold
overall_tpr = true_positive_rate(y_val_series, yhat_val_dnn)
overall_tpr

In [ ]:
# Compute group-specific thresholds
groups = sorted(A_val_series.unique())
thresholds = {}

for g in groups:
    idx = (A_val_series == g).to_numpy()
    best_g = threshold_to_hit_tpr(y_val_series[idx].to_numpy(), p_val_dnn[idx].to_numpy(), target_tpr=overall_tpr)
    thresholds[g] = best_g["t"]

thresholds

In [ ]:
# Apply group thresholds and evaluate
yhat_group = np.zeros(len(y_val_series), dtype=int)
for g in groups:
    idx = (A_val_series == g).to_numpy()
    yhat_group[idx] = (p_val_dnn[idx].to_numpy() >= thresholds[g]).astype(int)

mf_post = MetricFrame(
    metrics=metrics,
    y_true=y_val_series,
    y_pred=yhat_group,
    sensitive_features=A_val_series
)

print("Original DNN by-group:\n", mf.by_group)
print("\nPost-processed (TPR-matched) by-group:\n", mf_post.by_group)
print("\nTotal validation loss (original):", expected_cost(y_val_series, yhat_val_dnn))
print("Total validation loss (post-processed):", expected_cost(y_val_series, yhat_group))

**Question 7.** Compare the two rules on validation data. How do outreach rates, error rates, and total loss change by rule? Discuss the trade-offs of using recorded sex at the decision stage. Propose another approach, such as improving measurement or testing a model with a different feature set. What evidence would you need before recommending a pilot?

*Your answer:*

### Extension. Does the outreach rule need recorded sex?

**Question 8.** Fit one alternative DNN that excludes `SEX` from prediction. Keep the same observations, network settings, and training procedure. Retain the recorded groups separately for the fairness audit. Explain why removing the feature could affect predictive performance and why disparities might persist. Explain your modelling choices and interpret!

*Your answer:*

## Part 2. Digitizing Historical Economic Records

### Economic context

Historical price schedules and wage ledgers can help researchers measure living standards and labor market change. Building those data requires transcribing handwritten entries. A wrong digit can affect a measured wage or price, while checking every entry uses scarce archival staff time.

Use MNIST as a controlled benchmark for one step in this process: classifying an isolated handwritten digit from 0 to 9. Actual ledger entries would also pose challenges such as locating digits, reading multi-digit fields, and handling unfamiliar handwriting or damaged pages.

### Social cost and review capacity

For each digit, accept the model's transcription or send it to manual review. Assume that review returns the correct label and costs 0.06 units per digit. An incorrect automatic transcription costs 3.00 units. These are illustrative weights in a common unit of resource and data-quality loss.

Use all-manual review as an accounting benchmark. An automatically transcribed digit saves 0.06 units of review time and incurs 3.00 units if incorrect. A manually reviewed digit contributes zero savings relative to the benchmark. Staff can review at most 25% of digits, so all-manual review is infeasible under the stated capacity.

Compare a linear classifier and a CNN. Choose models and review policies on validation data before evaluating them on the official MNIST test set.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

# Data are MNIST from Keras
from tensorflow.keras.datasets import mnist

# For the baseline classifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

## Baseline code: load MNIST and visualize digits

In [ ]:
# Load MNIST
# x_train, x_test: uint8 images with shape (n, 28, 28)
# mnist_y_train, mnist_y_test: labels 0..9
(x_train, mnist_y_train), (x_test, mnist_y_test) = mnist.load_data()

x_train, x_val, mnist_y_train, mnist_y_val = train_test_split(
    x_train, mnist_y_train, test_size=0.10, stratify=mnist_y_train, random_state=42
)

print("x_train:", x_train.shape, x_train.dtype)
print("mnist_y_train:", mnist_y_train.shape, mnist_y_train.dtype)
print("x_test: ", x_test.shape,  x_test.dtype)
print("mnist_y_test: ", mnist_y_test.shape,  mnist_y_test.dtype)

# Show a few digits
mnist_rng = np.random.default_rng(0)
idx = mnist_rng.choice(len(x_train), size=12, replace=False)

plt.figure(figsize=(10, 4))
for i, j in enumerate(idx, start=1):
    plt.subplot(3, 4, i)
    plt.imshow(x_train[j], cmap="gray")
    plt.title(f"Label: {mnist_y_train[j]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Baseline: Logistic regression (linear classifier) ---

# Preprocess: flatten images and scale to [0, 1]
mnist_X_train = x_train.reshape(len(x_train), -1).astype(np.float32) / 255.0
mnist_X_val  = x_val.reshape(len(x_val), -1).astype(np.float32) / 255.0

# OPTIONAL: to make experiments run fast on laptops, you can subsample the training set.
# Set n_train = len(mnist_X_train) to use the full fitting split.
n_train = 15000
mnist_rng = np.random.default_rng(1)
sub_idx = mnist_rng.choice(len(mnist_X_train), size=n_train, replace=False)

X_train_sub = mnist_X_train[sub_idx]
y_train_sub = mnist_y_train[sub_idx]

# Logistic regression via SGD (one-vs-rest)
# - max_iter kept small on purpose to keep the baseline simple and fast
# - You are welcome to tune these later
logit = SGDClassifier(
    loss="log_loss",
    penalty="l2",
    alpha=1e-4,
    max_iter=10,
    tol=1e-3,
    random_state=0,
)

logit.fit(X_train_sub, y_train_sub)

# Predictions and accuracy
yhat_val = logit.predict(mnist_X_val)
baseline_val_acc = accuracy_score(mnist_y_val, yhat_val)
print(f"Baseline logistic val accuracy: {baseline_val_acc:.4f}")

# Confusion matrix
cm = confusion_matrix(mnist_y_val, yhat_val, labels=np.arange(10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))
disp.plot(values_format="d")
plt.title("Baseline (Logistic) Confusion Matrix")
plt.show()

# Show a few misclassified examples
mis = np.where(yhat_val != mnist_y_val)[0]
print("Number of val mistakes:", len(mis))

show_k = 12
mis_idx = mnist_rng.choice(mis, size=min(show_k, len(mis)), replace=False)

plt.figure(figsize=(10, 4))
for i, j in enumerate(mis_idx, start=1):
    plt.subplot(3, 4, i)
    plt.imshow(x_val[j], cmap="gray")
    plt.title(f"True {mnist_y_val[j]} / Pred {yhat_val[j]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

# We'll also keep predicted probabilities for later triage analysis.
# (SGDClassifier exposes predict_proba after fitting with loss='log_loss'.)
baseline_val_proba = logit.predict_proba(mnist_X_val)  # shape (n_val, 10)
baseline_pmax = baseline_val_proba.max(axis=1)

**Question 1.** Explain what flattening and scaling do to the pixel data. How does the linear classifier use pixel positions? Which digits are most often confused in the validation examples, and how might shifts, rotations, or scale changes affect a transcription system?

*Your answer:*

In [ ]:
# --- Baseline triage policy evaluation ---

# We already computed:
# - yhat_val (predicted class)
# - baseline_val_proba (class probabilities)
# - baseline_pmax (max predicted probability)

manual_cost = 0.06
error_cost = 3.00

def evaluate_policy(y_true, y_pred, pmax, tau, manual_cost=0.06, error_cost=3.00):
    """Return metrics for threshold-based triage policy."""
    auto = pmax >= tau
    manual = ~auto

    manual_rate = manual.mean()
    auto_rate = auto.mean()

    # Accuracy among auto-processed cases
    if auto.sum() > 0:
        auto_acc = (y_pred[auto] == y_true[auto]).mean()
        auto_err = 1 - auto_acc
    else:
        auto_acc = np.nan
        auto_err = np.nan

    # Incremental value relative to all-manual baseline:
    # - if auto & correct: +manual_cost
    # - if auto & incorrect: +manual_cost - error_cost
    correct_auto = auto & (y_pred == y_true)
    incorrect_auto = auto & (y_pred != y_true)

    inc_value = (
        correct_auto.mean() * manual_cost
        + incorrect_auto.mean() * (manual_cost - error_cost)
    )

    # Scale up to per 10,000 digits for readability
    inc_value_per_10k = inc_value * 10000

    return {
        "tau": tau,
        "manual_rate": manual_rate,
        "auto_rate": auto_rate,
        "auto_acc": auto_acc,
        "inc_value_per_10k": inc_value_per_10k,
    }

taus = np.r_[0.0, np.linspace(0.50, 0.99, 50)]
rows = [evaluate_policy(mnist_y_val, yhat_val, baseline_pmax, tau, manual_cost, error_cost) for tau in taus]

# Print best feasible tau (manual_rate <= 25%)
feasible = [r for r in rows if r["manual_rate"] <= 0.25]
best = max(feasible, key=lambda r: r["inc_value_per_10k"]) if feasible else None

if best is None:
    print("No thresholds satisfied the manual review constraint.")
else:
    print("Best feasible policy (baseline logistic):")
    for k, v in best.items():
        if k in {"tau"}:
            print(f"  {k:>16}: {v:.3f}")
        elif k.endswith("rate") or k.endswith("acc"):
            print(f"  {k:>16}: {v:.3%}")
        else:
            print(f"  {k:>16}: {v:,.2f}")

# Simple plot: incremental value vs manual rate
manual_rates = np.array([r["manual_rate"] for r in rows])
values_10k = np.array([r["inc_value_per_10k"] for r in rows])

plt.figure(figsize=(7,4))
plt.plot(manual_rates, values_10k)
plt.axvline(0.25, linestyle="--")
plt.xlabel("Manual review rate")
plt.ylabel("Net savings per 10,000 digits (cost units)")
plt.title("Baseline Logistic: Net Savings vs Manual Review Rate")
plt.show()

**Question 2.** Explain the review rule. It accepts a transcription when the model's maximum class probability is at least the threshold and routes the remaining digits to reviewers. Use the validation results to explain how coverage, transcription errors, review capacity, and net savings interact. Why could higher overall accuracy still yield lower net savings? What assumption about probability calibration would you need to interpret a score of 0.95 as a 95% chance of a correct transcription?

*Your answer:*

### Guided introduction to CNNs

Think of a **filter** as a small window that slides over an image. A 3 × 3 filter looks at nine neighboring pixels at a time. Its weights are learned during training and reused at every location, so the same filter can respond to a stroke wherever it appears. The grid of responses is called a **feature map**. A 2 × 2 **max pooling** layer keeps the strongest response in each 2 × 2 block, making the map smaller and less sensitive to small shifts within a block.

**Question 3.** Imagine a filter that responds to a short vertical stroke. What happens to its feature map if the stroke moves slightly? Explain what max pooling keeps and what detail it discards. Relate your answer to a handwritten digit of your choice.

*Your answer:*

### Building and fitting a CNN with Keras

Keras is the model-building interface we use through TensorFlow. `layers.Input(shape=(28, 28, 1))` describes one image with one grayscale channel. `Conv2D` learns filters for local patterns, `MaxPooling2D` shrinks the spatial grid, and the final softmax produces ten class probabilities. `compile` sets the optimizer, loss, and tracked metrics. `fit` trains the model, and `predict` returns one probability vector per image.

**Coding tip.** Use `cnn.summary()` to follow the changes in shape and `history.history["val_accuracy"]` to see validation accuracy by epoch. `.argmax(axis=1)` gives the predicted digit and `.max(axis=1)` gives its score. [TensorFlow CNN tutorial](https://www.tensorflow.org/tutorials/images/cnn)

In [ ]:
# --- Small CNN starter model ---

import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.utils.set_random_seed(42)

# Preprocess for CNN: add channel dimension and scale to [0, 1]
X_train_cnn = x_train.astype(np.float32) / 255.0
X_val_cnn  = x_val.astype(np.float32) / 255.0

X_train_cnn = X_train_cnn[..., np.newaxis]  # (n, 28, 28, 1)
X_val_cnn  = X_val_cnn[..., np.newaxis]

# Use the same training observations as the linear baseline.
sub_idx_cnn = sub_idx

X_train_cnn_sub = X_train_cnn[sub_idx_cnn]
y_train_cnn_sub = mnist_y_train[sub_idx_cnn]

cnn = models.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(10, activation="softmax")
])

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.summary()

history = cnn.fit(
    X_train_cnn_sub, y_train_cnn_sub,
    validation_data=(X_val_cnn, mnist_y_val),
    epochs=3,
    batch_size=128,
    verbose=1
)

# CNN val accuracy
cnn_val_probs = cnn.predict(X_val_cnn, verbose=0)
cnn_val_pred = cnn_val_probs.argmax(axis=1)
cnn_val_acc = accuracy_score(mnist_y_val, cnn_val_pred)
print(f"CNN val accuracy: {cnn_val_acc:.4f}")

# Compare to baseline logistic regression
print(f"Baseline logistic val accuracy: {baseline_val_acc:.4f}")
print(f"Accuracy improvement (CNN - baseline): {cnn_val_acc - baseline_val_acc:.4f}")

# Optional: confusion matrix for CNN
cm_cnn = confusion_matrix(mnist_y_val, cnn_val_pred, labels=np.arange(10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_cnn, display_labels=np.arange(10))
disp.plot(values_format="d")
plt.title("CNN Confusion Matrix")
plt.show()

**Question 4.** Use `cnn.summary()` to trace one image through the starter CNN. Record the height, width, and number of feature maps after each convolution and pooling layer. Why does the starter CNN flatten its input later than the linear classifier? Identify one training setting and one regularization choice in the code.

**Hint.** Start with a 28 × 28 image with one grayscale channel. With `padding="same"` and stride 1, a convolution keeps height and width fixed and produces one feature map per filter. The first convolution therefore outputs 28 × 28 × 16. Each 2 × 2 pooling layer halves the height and width in this model. Ignore the batch dimension in the summary.

*Your answer:*

### Extension. Allocating scarce manual review

**Question 5.** Design one CNN variant by changing a single architectural choice, such as filter counts or dropout. Explain what you expect the change to do. Keep the training observations, validation split, and training budget fixed. Compare validation accuracy for the linear baseline, starter CNN, and variant, and inspect one pair of digits that a CNN frequently confuses.

*Your answer:*

**Question 6.** For each CNN, use validation predictions to choose the threshold with the largest net savings among policies that review at most 25% of digits. Report review rate, accuracy among accepted digits, and net savings for both rules. Does the CNN with higher overall accuracy also give higher net savings? Explain which rule you would choose.

Freeze your chosen model and threshold, then evaluate them once on the official test set. Report the same three metrics and flag any capacity shortfall.

*Your answer:*

**Question 7.** In a short paragraph, give one example of how a transcription error could distort a statistic calculated from historical wage or price records. What would you check on a sample of real records before using the rule?

*Your answer:*

**Concluding Reflection:** What did this homework teach you about choosing a machine-learning model for an economic application? In a few sentences, identify what you would consider when choosing a model and explain why.

*Your answer:*

I hope you enjoyed learning about neural networks! Congrats for finishing homework 2!